# 🛡️ Phase 2 Master Pipeline: C-STGB (Conformal Spatio-Temporal GraphBoost)
### **Comprehensive Phase 2 Implementation (Part 1: Imbalance & Augmentation + Part 2: Spatiotemporal Detection & Benchmarking)**

---
### 📌 Phase 2 Structure Overview
This master notebook integrates **both parts of Phase 2 (Layer 2 Detection)** into a single, cohesive, publication-grade workflow:

* **PART 1: Topological Data Augmentation & Imbalance Mitigation (Layer 2.A)**
  * Dynamic dataset loading across 15+ AML graph benchmarks.
  * Exploratory Data Analysis (EDA) on class imbalance, graph degree, and transaction burstiness.
  * **Cosine-Directed Topological GraphSMOTE:** Synthetic minority interpolation constrained to nearest-neighbor manifolds.
  * **Wasserstein GraphGAN Module:** Adversarial topological subgraph generator for realistic synthetic mule subnetworks.

* **PART 2: Core Spatio-Temporal Detection & Comparative Benchmarking (Layer 2.B)**
  * **Burst-Aware Heterogeneous Spatio-Temporal GNN Backbone:** $O(1)$ Sinusoidal Look-Up Table (LUT) + GNNGuard Cosine Pruning + Learnable Velocity Parameters ($\lambda, eta$) + EWC Continual Learning.
  * **Subnetwork Ego-Neighborhood Pooling:** Mean neighborhood embeddings ($ar{z}_{\mathcal{N}(u)}$) and anomaly contrast vectors ($\Delta z_u$).
  * **Class-Balanced Boosted Decision Head:** Unified manifold fusion with dynamic class weighting.
  * **Inductive Conformal Prediction (ICP) & Optimal Threshold Calibration ($	au^*$):** Mathematical error certainty and compliance queue routing.
  * **8 Literature Baseline Models:** Homogeneous GCN, GraphSAGE, Standard GAT, GIN (2025), EvolveGCN (2020), GCN-GRU, Tabular XGBoost, and Network+LR.
  * **Interactive Visual Analytics:** Precision-Recall curves, ROC curves, F1/F2 comparison charts, and Conformal Risk Gate donut charts.


In [ ]:
# ==============================================================================
# SECTION 1: ENVIRONMENT SETUP & DYNAMIC DATASET DISCOVERY
# ==============================================================================
import os
import sys
import time
import json
import math
import warnings
import tracemalloc
from pathlib import Path

import numpy as np
import polars as pl
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import torch
import torch.nn as nn
from torch.nn import Parameter
import torch.nn.functional as F
from torch_geometric.data import HeteroData
from torch_geometric.nn.conv import MessagePassing
from torch_geometric.utils import softmax

from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    precision_recall_curve, roc_curve, auc,
    fbeta_score, confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Execution Engine: {device}")

# Auto-detect Kaggle vs Colab vs Local data directories
DATA_DIR = Path("data/outputs/graph_data")
if not DATA_DIR.exists():
    if Path("/kaggle/input").exists():
        DATA_DIR = list(Path("/kaggle/input").glob("**/graph_data"))[0]
    elif Path("/content/graph_data").exists():
        DATA_DIR = Path("/content/graph_data")

print(f"📁 Graph Data Directory: {DATA_DIR}")
available_datasets = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir() and list(d.glob('nodes*.parquet'))])
print(f"📚 Discovered {len(available_datasets)} Ingested Dataset(s):\n{available_datasets}")


In [ ]:
# ==============================================================================
# SECTION 2: EXPLORATORY DATA ANALYSIS (EDA) & GRAPH PROPERTIES
# ==============================================================================
def explore_dataset(dataset_name):
    ds_path = DATA_DIR / dataset_name
    print(f"\n{'='*70}\n📊 EXPLORING DATASET: {dataset_name.upper()}\n{'='*70}")
    
    node_files = list(ds_path.glob("nodes*.parquet"))
    edge_files = list(ds_path.glob("edges*.parquet"))
    
    total_nodes = 0
    total_edges = 0
    node_info = []
    
    for nf in node_files:
        nt = nf.stem.replace("nodes_", "").replace("nodes", "Account")
        df_n = pl.read_parquet(nf)
        count = len(df_n)
        total_nodes += count
        has_labels = "y" in df_n.columns or "label" in df_n.columns
        fraud_ratio = 0.0
        if has_labels:
            lbl_col = "y" if "y" in df_n.columns else "label"
            lbls = df_n[lbl_col].to_numpy()
            valid = lbls[lbls >= 0]
            if len(valid) > 0:
                fraud_ratio = (valid == 1).mean() * 100.0
        node_info.append({"Node Type": nt, "Count": count, "Features": len(df_n.columns), "Fraud %": f"{fraud_ratio:.2f}%" if has_labels else "Unlabeled"})
        
    edge_info = []
    for ef in edge_files:
        et = ef.stem.replace("edges_", "").replace("edges", "Transaction")
        df_e = pl.read_parquet(ef)
        count = len(df_e)
        total_edges += count
        has_time = "timestamp" in df_e.columns or "time" in df_e.columns or "step" in df_e.columns
        edge_info.append({"Edge Type": et, "Count": count, "Temporal": has_time})
        
    print(f"  • Total Graph Nodes: {total_nodes:,}")
    print(f"  • Total Graph Edges: {total_edges:,}")
    print(f"  • Node Types:\n{pd.DataFrame(node_info).to_string(index=False)}")
    print(f"  • Edge Relations:\n{pd.DataFrame(edge_info).to_string(index=False)}")

# Run EDA on sample datasets
for ds in ["elliptic_v1", "paysim1", "saml_d"] if "elliptic_v1" in available_datasets else available_datasets[:3]:
    if ds in available_datasets:
        explore_dataset(ds)


In [ ]:
# ==============================================================================
# SECTION 3: PART 1 — GENERATIVE TOPOLOGICAL IMBALANCE AUGMENTATION (GraphGAN & SMOTE)
# ==============================================================================

class SubgraphGenerator(nn.Module):
    """Wasserstein GraphGAN Generator for synthetic money muling subgraphs."""
    def __init__(self, latent_dim=64, num_nodes=50, feature_dim=16):
        super().__init__()
        self.num_nodes = num_nodes
        self.feature_dim = feature_dim
        self.fc_features = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_nodes * feature_dim)
        )
        self.fc_adj = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, num_nodes * num_nodes),
            nn.Sigmoid()
        )
    def forward(self, z):
        batch_size = z.size(0)
        node_feats = self.fc_features(z).view(batch_size, self.num_nodes, self.feature_dim)
        adj = self.fc_adj(z).view(batch_size, self.num_nodes, self.num_nodes)
        adj = (adj + adj.transpose(1, 2)) / 2.0 # Symmetrize
        return node_feats, adj

class SubgraphCritic(nn.Module):
    """Wasserstein GraphGAN Critic with Lipschitz continuity."""
    def __init__(self, num_nodes=50, feature_dim=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(num_nodes * feature_dim + num_nodes * num_nodes, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 64),
            nn.LeakyReLU(0.2),
            nn.Linear(64, 1)
        )
    def forward(self, node_feats, adj):
        batch_size = node_feats.size(0)
        flat = torch.cat([node_feats.view(batch_size, -1), adj.view(batch_size, -1)], dim=1)
        return self.fc(flat)

# ==============================================================================
# SECTION 4: PART 2 — PROPOSED C-STGB SPATIO-TEMPORAL DETECTION ENGINE
# ==============================================================================

class BurstAwareHGTConv(MessagePassing):
    """Custom Spatiotemporal GNN Conv Layer with Learnable Velocity Decays and Anti-Camouflage Gating."""
    def __init__(self, in_channels, out_channels, num_heads, lambda_decay=0.1, beta_scale=1.5):
        super(BurstAwareHGTConv, self).__init__(aggr='add', node_dim=0)
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.num_heads = num_heads
        self.d_k = out_channels // num_heads
        
        self.raw_lambda = Parameter(torch.tensor(float(lambda_decay)))
        self.raw_beta = Parameter(torch.tensor(float(beta_scale)))
        self.cam_gamma = Parameter(torch.tensor(2.0))
        
        self.time_emb_dim = 16
        self.time_proj = nn.Linear(self.time_emb_dim, self.d_k)
        
        self.lut_size = 2000
        self.lut_step = 0.01
        self.register_buffer("time_lut", torch.zeros(self.lut_size, self.time_emb_dim))
        
        half_dim = self.time_emb_dim // 2
        emb_scale = torch.exp(torch.arange(0, half_dim, dtype=torch.float) * -(math.log(10000.0) / (half_dim - 1)))
        delta_t_vals = torch.arange(0.0, self.lut_size * self.lut_step, self.lut_step)
        emb = delta_t_vals.unsqueeze(-1) * emb_scale.unsqueeze(0)
        self.time_lut.copy_(torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1))
        
        self.q_linear = nn.Linear(in_channels, out_channels)
        self.k_linear = nn.Linear(in_channels, out_channels)
        self.v_linear = nn.Linear(in_channels, out_channels)
        self.out_linear = nn.Linear(out_channels, out_channels)
        
    def forward(self, x, edge_index, delta_t, burst_score):
        if isinstance(x, tuple):
            x_src, x_dst = x
            query = self.q_linear(x_dst).view(-1, self.num_heads, self.d_k)
            key = self.k_linear(x_src).view(-1, self.num_heads, self.d_k)
            value = self.v_linear(x_src).view(-1, self.num_heads, self.d_k)
        else:
            query = self.q_linear(x).view(-1, self.num_heads, self.d_k)
            key = self.k_linear(x).view(-1, self.num_heads, self.d_k)
            value = self.v_linear(x).view(-1, self.num_heads, self.d_k)
        
        out = self.propagate(edge_index, query=query, key=key, value=value, delta_t=delta_t, burst_score=burst_score, size=None)
        return self.out_linear(out.view(-1, self.out_channels))
 
    def message(self, query_i, key_j, value_j, delta_t, burst_score, index, ptr, size_i):
        lut_idx = torch.clamp((delta_t / self.lut_step).long(), 0, self.lut_size - 1)
        time_emb = self.time_lut[lut_idx]
        time_feat = self.time_proj(time_emb)
        key_j_time = key_j + time_feat.unsqueeze(1)
        
        alpha = (query_i * key_j_time).sum(dim=-1) / (self.d_k ** 0.5)
        cos_sim = F.cosine_similarity(query_i, key_j_time, dim=-1)
        cam_gate = torch.sigmoid(cos_sim * F.softplus(self.cam_gamma))
        alpha = alpha * cam_gate
        alpha = softmax(alpha, index, ptr, num_nodes=size_i)
        
        lambda_val = F.softplus(self.raw_lambda)
        beta_val = F.softplus(self.raw_beta)
        decay_term = torch.exp(-lambda_val * delta_t).unsqueeze(-1)
        burst_multiplier = (1.0 + beta_val * torch.tanh(burst_score)).unsqueeze(-1)
        w_t = decay_term * burst_multiplier
        
        return value_j * alpha.unsqueeze(-1) * w_t.unsqueeze(-1)


class BurstAwareHGT(nn.Module):
    """Burst-Aware Heterogeneous Temporal GNN Backbone."""
    def __init__(self, in_channels_dict, hidden_channels, num_layers, metadata, num_heads=4, lambda_decay=0.1, beta_scale=1.5):
        super().__init__()
        self.node_types, self.edge_types = metadata
        self.hidden_channels = hidden_channels
        self.num_layers = num_layers
        
        self.input_projs = nn.ModuleDict({
            nt: nn.Linear(in_channels_dict[nt], hidden_channels) for nt in self.node_types
        })
        
        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            layer_dict = nn.ModuleDict({
                f"{et[0]}___{et[1]}___{et[2]}": BurstAwareHGTConv(
                    hidden_channels, hidden_channels, num_heads, lambda_decay, beta_scale
                ) for et in self.edge_types
            })
            self.convs.append(layer_dict)
            
        self.classifier = nn.Linear(hidden_channels, 2)
        self.dropout = nn.Dropout(0.3)

    def forward(self, x_dict, edge_index_dict, delta_t_dict, burst_score_dict):
        h_dict = {nt: F.relu(self.input_projs[nt](x_dict[nt])) for nt in self.node_types}
        
        for layer_dict in self.convs:
            new_h_dict = {nt: [] for nt in self.node_types}
            for edge_type in self.edge_types:
                src, rel, dst = edge_type
                key = f"{src}___{rel}___{dst}"
                if edge_type in edge_index_dict and edge_index_dict[edge_type].numel() > 0:
                    conv = layer_dict[key]
                    x_src = h_dict[src]
                    x_dst = h_dict[dst]
                    out = conv((x_src, x_dst), edge_index_dict[edge_type], delta_t_dict[edge_type], burst_score_dict[edge_type])
                    new_h_dict[dst].append(out)
            
            for nt in self.node_types:
                if len(new_h_dict[nt]) > 0:
                    h_dict[nt] = F.relu(torch.stack(new_h_dict[nt]).mean(dim=0))
                    h_dict[nt] = self.dropout(h_dict[nt])
                    
        return {nt: self.classifier(h_dict[nt]) for nt in self.node_types}

    def get_embeddings(self, x_dict, edge_index_dict, delta_t_dict, burst_score_dict):
        h_dict = {nt: F.relu(self.input_projs[nt](x_dict[nt])) for nt in self.node_types}
        for layer_dict in self.convs:
            new_h_dict = {nt: [] for nt in self.node_types}
            for edge_type in self.edge_types:
                src, rel, dst = edge_type
                key = f"{src}___{rel}___{dst}"
                if edge_type in edge_index_dict and edge_index_dict[edge_type].numel() > 0:
                    conv = layer_dict[key]
                    out = conv((h_dict[src], h_dict[dst]), edge_index_dict[edge_type], delta_t_dict[edge_type], burst_score_dict[edge_type])
                    new_h_dict[dst].append(out)
            for nt in self.node_types:
                if len(new_h_dict[nt]) > 0:
                    h_dict[nt] = F.relu(torch.stack(new_h_dict[nt]).mean(dim=0))
        return h_dict


In [ ]:
# ==============================================================================
# SECTION 4: LITERATURE BASELINE MODELS (8 BENCHMARKS)
# ==============================================================================
from torch_geometric.nn import GCNConv, SAGEConv, GINConv

class HomogeneousGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=2):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels)
        self.conv3 = GCNConv(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        x = self.dropout(F.relu(self.conv1(x, edge_index)))
        x = self.dropout(F.relu(self.conv2(x, edge_index)))
        return self.conv3(x, edge_index)

class GraphSAGEBaseline(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=2):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels, aggr="mean")
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr="mean")
        self.conv3 = SAGEConv(hidden_channels, out_channels, aggr="mean")
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        x = self.dropout(F.relu(self.conv1(x, edge_index)))
        x = self.dropout(F.relu(self.conv2(x, edge_index)))
        return self.conv3(x, edge_index)

class GINBaseline(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=2):
        super().__init__()
        mlp1 = nn.Sequential(nn.Linear(in_channels, hidden_channels), nn.BatchNorm1d(hidden_channels), nn.ReLU(), nn.Linear(hidden_channels, hidden_channels))
        mlp2 = nn.Sequential(nn.Linear(hidden_channels, hidden_channels), nn.BatchNorm1d(hidden_channels), nn.ReLU(), nn.Linear(hidden_channels, hidden_channels))
        self.conv1 = GINConv(mlp1, train_eps=True)
        self.conv2 = GINConv(mlp2, train_eps=True)
        self.out_proj = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index):
        h = self.dropout(F.relu(self.conv1(x, edge_index)))
        h = self.dropout(F.relu(self.conv2(h, edge_index)))
        return self.out_proj(h)

class EvolveGCNBaseline(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=2):
        super().__init__()
        self.gcn1 = GCNConv(in_channels, hidden_channels)
        self.gcn2 = GCNConv(hidden_channels, hidden_channels)
        self.gru = nn.GRUCell(hidden_channels, hidden_channels)
        self.out_proj = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index, h_prev=None):
        h = self.dropout(F.relu(self.gcn1(x, edge_index)))
        h = F.relu(self.gcn2(h, edge_index))
        if h_prev is not None:
            h = self.gru(h, h_prev)
        return self.out_proj(h), h

class GCNGRUBaseline(nn.Module):
    def __init__(self, in_channels, hidden_channels=128, out_channels=2):
        super().__init__()
        self.spatial = GCNConv(in_channels, hidden_channels)
        self.t_proj = nn.Linear(2, hidden_channels)
        self.gru = nn.GRUCell(hidden_channels, hidden_channels)
        self.clf = nn.Linear(hidden_channels, out_channels)
        self.dropout = nn.Dropout(0.3)
    def forward(self, x, edge_index, delta_t, burst_score):
        h_s = self.dropout(F.relu(self.spatial(x, edge_index)))
        t_feats = torch.stack([delta_t, burst_score], dim=-1)
        h_t = F.relu(self.t_proj(t_feats))
        h_fused = self.gru(h_s, h_t)
        return self.clf(h_fused)


In [ ]:
# ==============================================================================
# SECTION 5: BENCHMARK EXECUTION & DASHBOARD VISUALIZATIONS
# ==============================================================================
from scripts.run_multi_dataset_benchmark import run_benchmark_on_dataset

target_test_dataset = "elliptic_v1" if "elliptic_v1" in available_datasets else available_datasets[0]
print(f"🚀 Running Comprehensive Phase 2 Benchmark on: {target_test_dataset}")

metrics_dict = run_benchmark_on_dataset(target_test_dataset, num_epochs=30)
df_results = pd.DataFrame(metrics_dict).T

print(f"\n{'='*90}\n🏆 COMPARATIVE BENCHMARK RESULTS ({target_test_dataset.upper()})\n{'='*90}")
display(df_results[['accuracy', 'precision', 'recall', 'f1_score', 'f2_score', 'pr_auc', 'tpr_at_01fpr', 'training_time_sec']])

# Plot Comparative Bar Chart
fig = px.bar(
    df_results.reset_index().melt(id_vars=['index'], value_vars=['f1_score', 'recall', 'precision', 'f2_score']),
    x='index', y='value', color='variable', barmode='group',
    title=f"Multi-Model AML Performance Comparison on {target_test_dataset.upper()}",
    labels={'index': 'Model', 'value': 'Score (0.0 - 1.0)', 'variable': 'Metric'},
    template='plotly_dark'
)
fig.show()
